# Simple LDA

get Topics

https://radimrehurek.com/gensim/auto_examples/tutorials/run_lda.html

## Load Data

In [1]:
import io
import os.path
import re
import tarfile

import smart_open

def extract_documents(url='https://cs.nyu.edu/~roweis/data/nips12raw_str602.tgz'):
    with smart_open.open(url, "rb") as file:
        with tarfile.open(fileobj=file) as tar:
            for member in tar.getmembers():
                if member.isfile() and re.search(r'nipstxt/nips\d+/\d+\.txt', member.name):
                    member_bytes = tar.extractfile(member).read()
                    yield member_bytes.decode('utf-8', errors='replace')

docs = list(extract_documents())

In [2]:
print(len(docs))
print(docs[1])

1740
1 
CONNECTIVITY VERSUS ENTROPY 
Yaser S. Abu-Mostafa 
California Institute of Technology 
Pasadena, CA 91125 
ABSTRACT 
How does the connectivity of a neural network (number of synapses per 
neuron) relate to the complexity of the problems it can handle (measured by 
the entropy)? Switching theory would suggest no relation at all, since all Boolean 
functions can be implemented using a circuit with very low connectivity (e.g., 
using two-input NAND gates). However, for a network that learns a problem 
from examples using a local learning rule, we prove that the entropy of the 
problem becomes a lower bound for the connectivity of the network. 
INTRODUCTION 
The most distinguishing feature of neural networks is their ability to spon- 
taneously learn the desired function from 'training' samples, i.e., their ability 
to program themselves. Clearly, a given neural network cannot just learn any 
function, there must be some restrictions on which networks can learn which 
functions. On

# NLTK Stop words
from nltk.corpus import stopwords
stop_words = stopwords.words('english')
stop_words.extend(['from', 'subject', 're', 'edu', 'use'])

## Create tokenized corpus

In [3]:
# Tokenize the documents

from nltk.tokenize import RegexpTokenizer

# Split the documents into tokens

tokenizer = RegexpTokenizer(r'\w+')
for idx in range(len(docs)):
    docs[idx] = docs[idx].lower()  # Convert to lowercase
    docs[idx] = tokenizer.tokenize(docs[idx])  # Split into words

# Remove unnecessary words

docs = [[token for token in doc if not token.isnumeric()] for doc in docs] # Remove numbers

docs = [[token for token in doc if len(token) > 1] for doc in docs] # Remove words with only one character

In [6]:
# Lemmatize the documents

import nltk
from nltk.corpus import wordnet as wn
from nltk.stem.wordnet import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
docs = [[lemmatizer.lemmatize(token) for token in doc] for doc in docs]

In [7]:
# Compute bigrams
from gensim.models import Phrases

# Add bigrams and trigrams to docs (only ones that appear 20 times or more)

bigram = Phrases(docs, min_count=20)
for idx in range(len(docs)):
    for token in bigram[docs[idx]]:
        if '_' in token:
            # Token is a bigram, add to document
            docs[idx].append(token)

In [8]:
# Remove rare and common tokens

from gensim.corpora import Dictionary

# Create a dictionary representation of the documents

dictionary = Dictionary(docs)

# Filter out words that occur less than 20 documents, or more than 50% of the documents

dictionary.filter_extremes(no_below=20, no_above=0.5)

In [9]:
# Bag-of-words representation of the documents

corpus = [dictionary.doc2bow(doc) for doc in docs]

In [10]:
print('Number of unique tokens: %d' % len(dictionary))
print('Number of documents: %d' % len(corpus))

Number of unique tokens: 8644
Number of documents: 1740


# Initialize and train LDA Model

In [11]:
# Train LDA model.
from gensim.models import LdaModel

# Set training parameters.
num_topics = 10
chunksize = 2000
passes = 20
iterations = 400
eval_every = None  # Don't evaluate model perplexity, takes too much time.

# Make an index to word dictionary.
temp = dictionary[0]  # This is only to "load" the dictionary.
id2word = dictionary.id2token

model = LdaModel(
    corpus=corpus,
    id2word=id2word,
    chunksize=chunksize,
    alpha='auto',
    eta='auto',
    iterations=iterations,
    num_topics=num_topics,
    passes=passes,
    eval_every=eval_every
)

# Get top 3 Keywords

In [31]:

for i in range (0, len(corpus)):
    topic_distribution = model.get_document_topics(corpus[i])
    # Sort the topics by their probability in descending order
    sorted_topics = sorted(topic_distribution, key=lambda x: x[1], reverse=True)
    # Get the top N keywords for the highest-probability topic
    top_keywords = model.show_topic(sorted_topics[0][0], topn=3)
    # Extract and return the keyword strings
    keywords = [keyword for keyword, _ in top_keywords]

    print(keywords)

['hidden', 'hidden_unit', 'layer']
['bound', 'let', 'theorem']
['neuron', 'memory', 'synaptic']
['bound', 'let', 'theorem']
['neuron', 'memory', 'synaptic']
['bound', 'let', 'theorem']
['hidden', 'hidden_unit', 'layer']
['cell', 'visual', 'field']
['hidden', 'hidden_unit', 'layer']
['spike', 'signal', 'neuron']
['spike', 'signal', 'neuron']
['spike', 'signal', 'neuron']
['neuron', 'memory', 'synaptic']
['image', 'object', 'chip']
['bound', 'let', 'theorem']
['recognition', 'speech', 'word']
['spike', 'signal', 'neuron']
['action', 'policy', 'control']
['bound', 'let', 'theorem']
['bound', 'let', 'theorem']
['neuron', 'memory', 'synaptic']
['neuron', 'memory', 'synaptic']
['neuron', 'memory', 'synaptic']
['hidden', 'hidden_unit', 'layer']
['control', 'field', 'dynamic']
['neuron', 'memory', 'synaptic']
['neuron', 'memory', 'synaptic']
['neuron', 'memory', 'synaptic']
['control', 'field', 'dynamic']
['spike', 'signal', 'neuron']
['bound', 'let', 'theorem']
['neuron', 'memory', 'synaptic'

# Sort Documents into Topics

In [12]:
top_topics = model.top_topics(corpus)

# Average topic coherence is the sum of topic coherences of all topics, divided by the number of topics.
avg_topic_coherence = sum([t[1] for t in top_topics]) / num_topics
print('Average topic coherence: %.4f.' % avg_topic_coherence)

from pprint import pprint
pprint(top_topics)

Average topic coherence: -1.1561.
[([(0.009407911, 'bound'),
   (0.007666053, 'let'),
   (0.006614542, 'theorem'),
   (0.0065245978, 'class'),
   (0.005315205, 'approximation'),
   (0.0049978113, 'threshold'),
   (0.0045501227, 'xi'),
   (0.0040522967, 'proof'),
   (0.004028036, 'dimension'),
   (0.0039932914, 'node'),
   (0.0039141094, 'sample'),
   (0.0037658643, 'loss'),
   (0.0036340868, 'polynomial'),
   (0.003322045, 'complexity'),
   (0.003214062, 'layer'),
   (0.0029947131, 'optimal'),
   (0.0028487367, 'net'),
   (0.0028462515, 'solution'),
   (0.0028200338, 'condition'),
   (0.0026891872, 'assume')],
  -0.9081491760120944),
 ([(0.02178919, 'cell'),
   (0.011919056, 'visual'),
   (0.011186311, 'field'),
   (0.009198805, 'direction'),
   (0.008750328, 'response'),
   (0.008597694, 'map'),
   (0.008551389, 'stimulus'),
   (0.0080467425, 'motion'),
   (0.007860003, 'orientation'),
   (0.0076068174, 'eye'),
   (0.007250286, 'neuron'),
   (0.007020244, 'receptive'),
   (0.006762570

In [16]:
import gensim

import pyLDAvis.gensim_models as gensimvis

pyLDAvis.enable_notebook()

gensimvis.prepare(model, corpus, dictionary)

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0     -0.093377 -0.023236       1        1  16.401654
4     -0.076408 -0.031410       2        1  11.616692
7     -0.118074  0.087897       3        1  11.228806
5     -0.082952  0.054220       4        1  10.522853
3      0.097078  0.072823       5        1  10.231625
1      0.049289 -0.110198       6        1   9.024975
9     -0.089139 -0.126279       7        1   8.165012
2      0.183856 -0.019802       8        1   8.151846
6     -0.029594  0.075377       9        1   7.329058
8      0.159321  0.020608      10        1   7.327480, topic_info=             Term         Freq        Total Category  logprob  loglift
247         image  7079.000000  7079.000000  Default  30.0000  30.0000
762        neuron  8505.000000  8505.000000  Default  29.0000  29.0000
630          cell  6141.000000  6141.000000  Default  28.0000  28.0000
413   recognition  4459.000000  4459.000000  Default  27.0000  27.0000
482        speech  2593.000000  2593.000000  Default  26.0000  26.0000
...           ...          ...          ...      ...      ...      ...
1828     activity   565.547736  3018.692775  Topic10  -5.5798   0.9388
186           fig   544.816315  3267.362362  Topic10  -5.6171   0.8222
1482    threshold   482.529929  2746.785610  Topic10  -5.7385   0.8744
1026    component   493.926050  3915.860472  Topic10  -5.7152   0.5431
1312        phase   458.780262  2343.506234  Topic10  -5.7890   0.9827

[728 rows x 6 columns], token_table=      Topic      Freq   Term
term                        
2150      1  0.043403     3d
2150      2  0.060764     3d
2150      3  0.014468     3d
2150      4  0.011574     3d
2150      6  0.729170     3d
...     ...       ...    ...
899       9  0.015111     xi
899      10  0.001511     xi
6885      1  0.041077  yi_xi
6885      3  0.903704  yi_xi
6885      4  0.041077  yi_xi

[3706 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 5, 8, 6, 4, 2, 10, 3, 7, 9])